In [71]:
import json
import pandas as pd

In [72]:
with open('../data/ner_result.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

In [73]:
df = pd.read_csv("../data/shinline_cafe_reviews_test.csv")

In [74]:
df

,사업장명,place_id,리뷰내용,작성일
0,어반커피,12066484,2층에 있는 커피숍!! 내부는 깨끗하고 친구들과 수다 떨기 넘 좋아요!! 쿠키 프라...,25.12.27.
1,어반커피,12066484,내부 깨끗. 친절하시고 음료 넘넘 맛있어요~~ 특히 아이스푸딩크림라떼!!! 가격도...,25.12.27.
2,어반커피,12066484,"말렌카는 맛있었는데,,음료는 좀 아쉬웠어요ㅠㅠ메뉴에 그려져있던 체리블라썸이랑은 넘 ...",25.8.31.
3,어반커피,12066484,커피 음료 가격이 비싼편이라 그런가 한가로웠어요 인테리어 이쁘고 커피맛도 괜찮았지...,25.7.4.
4,어반커피,12066484,진짜 최악입니다 디저트 자체 맛은 괜찮았는데 매장 관리가 아예 안되고 있는듯 자리에...,25.6.7.
...,...,...,...,...
15228,매머드 익스프레스 신림대학점,1746293546,작은 사이즈 있어서 좋아요,22.2.20.
15229,매머드 익스프레스 신림대학점,1746293546,친절하시고 커피가 저렴해서 좋아요,22.2.7.
15230,매머드 익스프레스 신림대학점,1746293546,딸기 요거트 스무디가 진짜 존맛,22.1.29.
15231,매머드 익스프레스 신림대학점,1746293546,가격이 저렴하다고 커피 맛이 안좋다는 편견을 깨주는 브랜드입니다 ㅎㅎㅎ,21.12.25.


In [75]:
data

{'7406': {'index': 7406,
  'original_text': '#신림카페#사장님센스굿#레몬청#카페맛집#수제카페#먹스타그램#먹부림#갬성#인스타맛집직접 수제로 만든 청과 디저트라서 더 담백하고 새콤달콤해요. 맛 보시면 사장님 정성 다 느껴집니다ㅎ꼭 한번 방문해보세요^^',
  'ner_tags': {'MENU': [{'entity': '레몬청',
     'descriptor': '담백하고 새콤달콤하다',
     'sentiment': '긍정'}],
   'FACILITY': [],
   'ATMOSPHERE': [],
   'TARGET': []}},
 '2767': {'index': 2767,
  'original_text': '커피가 연하고 맛있어용 ㅎ 강아지가 반겨줘요 ㅎ',
  'ner_tags': {'MENU': [{'entity': '커피',
     'descriptor': '맛있다',
     'sentiment': '긍정'}],
   'FACILITY': [],
   'ATMOSPHERE': [],
   'TARGET': []}},
 '2766': {'index': 2766,
  'original_text': '사장님도 친절하시고 강아지랑 같이 왓는데 비숑들이 반겨줘서 좋앗습니다!맛잇어요~',
  'ner_tags': {'MENU': [{'entity': '맛',
     'descriptor': '맛있다',
     'sentiment': '긍정'}],
   'FACILITY': [],
   'ATMOSPHERE': [],
   'TARGET': [{'entity': '강아지', 'descriptor': '좋다', 'sentiment': '긍정'}]}},
 '2768': {'index': 2768,
  'original_text': '오늘도 만족합니다! 어제리뷰써서 서비스 샐러드 받아서 먹었는데 오 맛있더라구요 여자친구도 맛있다고 그러더라구요 그래서 조심스럽게 사장님에게 더받을수있냐고 여쭤봤는데 된다고 하셔서요ㅠ 감사합니다!

In [76]:
df_store = df
ner_data = data
import pandas as pd
import ast

# ------------------------------------------------------------------
# 데이터 결합 및 평탄화 (문자열 요소를 자동으로 감지하고 처리)
# ------------------------------------------------------------------
parsed_rows = []
target_categories = ['FACILITY', 'ATMOSPHERE', 'TARGET']

for str_idx, val in ner_data.items():
    target_idx = int(val['index'])
    
    if target_idx in df_store.index:
        store_name = df_store.loc[target_idx, '사업장명']
        ner_tags = val.get('ner_tags', {})
        
        for category in target_categories:
            items = ner_tags.get(category, [])
            
            if isinstance(items, list):
                for item in items:
                    if isinstance(item, str):
                        item = item.strip()
                        if item.startswith('{') and item.endswith('}'):
                            try:
                                item = ast.literal_eval(item)
                            except (ValueError, SyntaxError):
                                print(item)
                                continue # 변환 실패 시 패스
                        else:
                            # 케이스 B: 진짜 단순 단어 문자열일 때 (예: '콘센트')
                            parsed_rows.append({
                                '사업장명': store_name,
                                '키워드': item,          # 단어 자체를 키워드로 저장
                                '키워드타입': category,   # ✨ [추가] 현재 카테고리 저장
                                'descriptor': '',       # 비어있으므로 공백 처리
                                'sentiment': ''         # 비어있으므로 공백 처리
                            })
                            continue
                    
                    # 케이스 C: 정상적으로 딕셔너리 형태로 저장되어 있거나 위의 변환이 성공했을 때
                    if isinstance(item, dict):
                        parsed_rows.append({
                            '사업장명': store_name,
                            '키워드': item.get('entity', ''),
                            '키워드타입': category,   # ✨ [추가] 현재 카테고리 저장
                            'descriptor': item.get('descriptor', ''),
                            'sentiment': item.get('sentiment', '')
                        })

# ------------------------------------------------------------------
# 최종 데이터프레임 빌드
# ------------------------------------------------------------------
df_final = pd.DataFrame(parsed_rows)

In [77]:
df_final

,사업장명,키워드,키워드타입,descriptor,sentiment
0,카페캘리 본점,강아지,TARGET,좋다,긍정
1,카페캘리 본점,여자친구,TARGET,맛있다고 하다,긍정
2,"?커피,MOCC",인테리어,FACILITY,아늑하다,긍정
3,"?커피,MOCC",주말마다 가고싶다,TARGET,None,긍정
4,"?커피,MOCC",카페,FACILITY,귀엽다,긍정
...,...,...,...,...,...
10611,궁전다방,인테리어,FACILITY,멋지다,긍정
10612,궁전다방,화장실,FACILITY,호텔 같다,긍정
10613,Mono Hertz(모노헤르쯔),화장실,FACILITY,깨끗하다,긍정
10614,Mono Hertz(모노헤르쯔),내부,FACILITY,좌석 다양하다,긍정


In [78]:
list(df_final[df_final['사업장명'] == '카페캘리 본점']['키워드'])

['강아지',
 '여자친구',
 '가성비',
 '가족들',
 '추천',
 '자주 방문',
 '캘리',
 '재방문',
 '미팅',
 '애견동반',
 '귀여운 비숑 강아지들',
 '반려견 동반',
 '강아지',
 '카페']

In [79]:
df[df['사업장명'] == '카페캘리 본점']

,사업장명,place_id,리뷰내용,작성일
2762,카페캘리 본점,1712594911,강아지와 산책 후 너무 추워서 방문했더니 따뜻하고 신림천이 내려다 보이는 뷰가 너무...,25.12.14.
2763,카페캘리 본점,1712594911,서원역 근처 오며 가며 보다가 애견동반해서 처음 방문했는데 강아지가 좋아해서 뿌듯하...,25.9.14.
2764,카페캘리 본점,1712594911,귀여운 비숑 강아지들이 왕왕 반겨주는 카페에요~ 반려견 동반 가능합니다~ 브런치와 ...,25.7.12.
2765,카페캘리 본점,1712594911,강아지도 데려올 수 있네요 친절하시고 리얼수박 맛있어요,24.7.27.
2766,카페캘리 본점,1712594911,첫방문해서 '☕️아메리카노 1리터ㆍ카페라떼' 테이크아웃 다녀왔는데요~ 🧏사장님이 정...,24.4.27.
2767,카페캘리 본점,1712594911,사장님도 친절하시고 강아지랑 같이 왓는데 비숑들이 반겨줘서 좋앗습니다!맛잇어요~❤️,24.4.27.
2768,카페캘리 본점,1712594911,커피가 연하고 맛있어용 ㅎㅎ 강아지가 반겨줘요 ㅎ,24.4.13.
2769,카페캘리 본점,1712594911,오늘도 만족합니다! 어제리뷰써서 서비스 샐러드 받아서 먹었는데 오 맛있더라구요 여자...,24.4.13.
2770,카페캘리 본점,1712594911,오늘도 방문해서 러스크랑 샐러드 서비스 받았어요~~자주방문할거같아요~~~!,24.4.12.
2771,카페캘리 본점,1712594911,1리터 아메리카노 테이크아웃이 1900원 이라길래 처음 방문했는데 진짜였어요!. 사...,24.4.11.


In [80]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import umap.umap_ as umap
import hdbscan

# 0. GPU 사용 가능 여부 확인 (비싼 서버가 일하고 있는지 체크!)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 현재 사용 중인 디바이스: {device.upper()}")

# 1. BGE-M3 모델 로드 (GPU에 할당)
print("모델 로딩 중...")
model = SentenceTransformer('dragonkue/BGE-m3-ko', device=device)

unique_keywords = df_final['키워드'].dropna().unique().tolist()
print(f"추출된 고유 키워드 개수: {len(unique_keywords)}개")

print("단어 임베딩 추출 중 (GPU 가속)...")
word_embeddings = model.encode(unique_keywords, batch_size=256, show_progress_bar=True)

# 5. UMAP 차원 축소 (CPU 연산)
print("UMAP 차원 축소 중...")
umap_model = umap.UMAP(n_neighbors=3, n_components=5, metric='cosine', random_state=42)
umap_embeddings = umap_model.fit_transform(word_embeddings)

# 6. HDBSCAN 클러스터링 (CPU 연산)
print("HDBSCAN 클러스터링 중...")
hdbscan_model = hdbscan.HDBSCAN(min_cluster_size=3, metric='euclidean', cluster_selection_method='eom')
clusters = hdbscan_model.fit_predict(umap_embeddings)

# 7. 결과 매핑
df_clusters_word = pd.DataFrame({
    '원본_키워드': unique_keywords,
    '군집_ID(단어기준)': clusters
})

print("\n[단어 기반 클러스터링 결과]")
print(df_clusters_word.sort_values(by='군집_ID(단어기준)'))

c:\VSCode\cafe_recomendation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
df_clusters_word[df_clusters_word['군집_ID(단어기준)'] == -1]

,원본_키워드,군집_ID(단어기준)
12,따뜻하고 아늑한 분위기,-1
15,따뜻하고 귀여운 분위기,-1
47,분위기,-1
63,작업겸 책 읽기,-1
70,프랜차이즈 커피,-1
...,...,...
3097,조용하고 넓고 깨끗한,-1
3117,많은 사람들과 함께하기,-1
3122,존맛,-1
3127,그릭요거트,-1


In [ ]:
# 4. 단어를 카페 도메인의 문장으로 확장
template_sentences = [f"이 카페의 핵심 특징은 {kw}입니다." for kw in unique_keywords]

# 5. 🔥 문장 기반 임베딩 (GPU 병렬 처리) 🔥
# 문장이 길어졌으므로 batch_size를 적절히 유지하며 GPU 연산을 태웁니다.
print("문장 임베딩 추출 중 (GPU 가속)...")
sentence_embeddings = model.encode(template_sentences, batch_size=256, show_progress_bar=True)

# 6. UMAP 차원 축소
print("UMAP 차원 축소 중...")
umap_model = umap.UMAP(n_neighbors=3, n_components=5, metric='cosine', random_state=42)
umap_embeddings = umap_model.fit_transform(sentence_embeddings)

# 7. HDBSCAN 클러스터링
print("HDBSCAN 클러스터링 중...")
hdbscan_model = hdbscan.HDBSCAN(min_cluster_size=3, metric='euclidean', cluster_selection_method='eom')
clusters = hdbscan_model.fit_predict(umap_embeddings)

# 8. 결과 매핑 (화면에는 원본 단어만 표시)
df_clusters_sentence = pd.DataFrame({
    '원본_키워드': unique_keywords, 
    '확장_문장(참고용)': template_sentences,
    '군집_ID(문장기준)': clusters
})

print("\n[문장 확장 기반 클러스터링 결과 (최종 매핑)]")
print(df_clusters_sentence[['원본_키워드', '군집_ID(문장기준)']].sort_values(by='군집_ID(문장기준)'))

문장 임베딩 추출 중 (GPU 가속)...


Batches: 100%|██████████| 13/13 [10:46<00:00, 49.75s/it]
/workspace/cafe_recomendation/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP 차원 축소 중...
HDBSCAN 클러스터링 중...

[문장 확장 기반 클러스터링 결과 (최종 매핑)]
       원본_키워드  군집_ID(문장기준)
2533       최고           -1
330   편안한 분위기           -1
320        가방           -1
2057  힐링되는 시간           -1
1651       도안           -1
...       ...          ...
3138   맞춤형 카페          370
2006   꼬수운 커피          370
2655     프리퀀시          370
1396   커피 클래스          370
2562      커피숍          370

[3139 rows x 2 columns]


In [ ]:
df_clusters_sentence[df_clusters_sentence['군집_ID(문장기준)'] == -1]

,원본_키워드,확장_문장(참고용),군집_ID(문장기준)
5,소량으로 판매하는게 아쉬워요,이 카페의 핵심 특징은 소량으로 판매하는게 아쉬워요입니다.,-1
16,추천,이 카페의 핵심 특징은 추천입니다.,-1
41,백색소음 느낌,이 카페의 핵심 특징은 백색소음 느낌입니다.,-1
42,꽃향기,이 카페의 핵심 특징은 꽃향기입니다.,-1
65,초록 뷰,이 카페의 핵심 특징은 초록 뷰입니다.,-1
...,...,...,...
3089,산쪽뷰,이 카페의 핵심 특징은 산쪽뷰입니다.,-1
3091,기분 좋은 하루,이 카페의 핵심 특징은 기분 좋은 하루입니다.,-1
3103,시즌 느낌,이 카페의 핵심 특징은 시즌 느낌입니다.,-1
3119,책장,이 카페의 핵심 특징은 책장입니다.,-1


In [ ]:
# df_clusters_word.to_csv('../data/ner_keyword_word_clt.csv',index=False, encoding="utf-8-sig")
# df_clusters_sentence.to_csv('../data/ner_keyword_sent_clt.csv',index=False, encoding="utf-8-sig")

# openpai로 확인하기

In [ ]:
df_clusters_word = pd.read_csv('../data/ner_keyword_word_clt.csv')
df_clusters_sentence = pd.read_csv('../data/ner_keyword_sent_clt.csv')

In [ ]:
import json
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
from prompt import cluter_system_prompt

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def get_representative(keywords) -> str | None:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": cluter_system_prompt},
            {"role": "user", "content": str(keywords)}
        ],
        response_format={"type": "json_object"}
    )
    parsed = json.loads(response.choices[0].message.content)
    return parsed["representative"]  # str 또는 None

def evaluate_clustering(df: pd.DataFrame, cluster_col: str, label: str) -> pd.DataFrame:
    grouped = df.groupby(cluster_col)["원본_키워드"].apply(list)

    results = {}
    for cluster_id, keywords in tqdm(grouped.items(), desc=f"[{label}] 대표값 생성"):
        results[cluster_id] = get_representative(keywords)

    df_rep = pd.DataFrame(
        list(results.items()),
        columns=[cluster_col, "대표_키워드"]
    )

    # 통계 출력
    total = len(df_rep)
    none_count = df_rep["대표_키워드"].isna().sum()
    valid_count = total - none_count
    print(f"\n{'='*40}")
    print(f"[{label}] 클러스터링 품질 평가")
    print(f"  전체 군집 수 : {total}")
    print(f"  대표값 성공  : {valid_count} ({valid_count/total*100:.1f}%)")
    print(f"  None (비일관): {none_count} ({none_count/total*100:.1f}%)")
    print(f"{'='*40}\n")

    return df.merge(df_rep, on=cluster_col, how="left")

# 두 클러스터링 방식 비교
df_result_word = evaluate_clustering(df_clusters_word, "군집_ID(단어기준)", "단어 기반")
df_result_sentence = evaluate_clustering(df_clusters_sentence, "군집_ID(문장기준)", "문장 기반")

[단어 기반] 대표값 생성: 378it [04:56,  1.27it/s]



[단어 기반] 클러스터링 품질 평가
  전체 군집 수 : 378
  대표값 성공  : 317 (83.9%)
  None (비일관): 61 (16.1%)



[문장 기반] 대표값 생성: 372it [04:44,  1.31it/s]


[문장 기반] 클러스터링 품질 평가
  전체 군집 수 : 372
  대표값 성공  : 326 (87.6%)
  None (비일관): 46 (12.4%)



In [ ]:
def add_recluster_column(df: pd.DataFrame, cluster_col: str = "군집 ID(문장기준)") -> pd.DataFrame:
    df = df.copy()
    df["재군집_키워드"] = None

    # 1. 확정 대표값 → 군집 내 모든 행에 대표_키워드 채우기
    confirmed_mask = df["대표_키워드"].notna() & (df[cluster_col] != -1)
    df.loc[confirmed_mask, "재군집_키워드"] = df.loc[confirmed_mask, "대표_키워드"]

    # 2. None 군집 → 원본_키워드
    none_mask = df["대표_키워드"].isna() & (df[cluster_col] != -1)
    df.loc[none_mask, "재군집_키워드"] = df.loc[none_mask, "원본_키워드"]

    # 3. -1 군집 → 원본_키워드
    noise_mask = df[cluster_col] == -1
    df.loc[noise_mask, "재군집_키워드"] = df.loc[noise_mask, "원본_키워드"]
    # 통계
    total = len(df)
    print(f"{'='*40}")
    print(f"  원본 전체            : {total}개")
    print(f"  확정 대표값 (unique) : {df[confirmed_mask]['재군집_키워드'].nunique()}개")
    print(f"  None군집 원본        : {int(none_mask.sum())}개")
    print(f"  노이즈(-1) 원본      : {int(noise_mask.sum())}개")
    print(f"  ───────────────────")
    print(f"  재군집_키워드 (unique): {df['재군집_키워드'].nunique()}개")  # nunique()는 NaN 자동 제외
    print(f"{'='*40}")

    return df

df_result_sentence = add_recluster_column(df_result_sentence, cluster_col="군집_ID(문장기준)")

  원본 전체            : 3139개
  확정 대표값 (unique) : 308개
  None군집 원본        : 326개
  노이즈(-1) 원본      : 309개
  ───────────────────
  재군집_키워드 (unique): 941개


In [ ]:
df_result_sentence

,원본_키워드,확장_문장(참고용),군집_ID(문장기준),대표_키워드,재군집_키워드
0,강아지,이 카페의 핵심 특징은 강아지입니다.,96,반려동물 카페,반려동물 카페
1,여자친구,이 카페의 핵심 특징은 여자친구입니다.,180,여자친구,여자친구
2,인테리어,이 카페의 핵심 특징은 인테리어입니다.,359,카페 인테리어,카페 인테리어
3,주말마다 가고싶다,이 카페의 핵심 특징은 주말마다 가고싶다입니다.,279,주말 카페,주말 카페
4,카페,이 카페의 핵심 특징은 카페입니다.,369,None,카페
...,...,...,...,...,...
3134,단골커피집,이 카페의 핵심 특징은 단골커피집입니다.,19,단골카페,단골카페
3135,컵디자인,이 카페의 핵심 특징은 컵디자인입니다.,363,컵,컵
3136,결제시스템,이 카페의 핵심 특징은 결제시스템입니다.,291,키오스크,키오스크
3137,홀더,이 카페의 핵심 특징은 홀더입니다.,44,홀더,홀더


In [ ]:
df_result_sentence.to_csv('../data/ner_keyword_sent_re_clt.csv',index=False, encoding="utf-8-sig")

# 재군집

In [ ]:
import pandas as pd
df_result_sentence = pd.read_csv("../data/ner_keyword_sent_re_clt.csv")

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import umap.umap_ as umap
import hdbscan

# 0. 디바이스 확인
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"현재 사용 중인 디바이스: {device.upper()}")

print("모델 로딩 중...")
model = SentenceTransformer('dragonkue/BGE-m3-ko', device=device)

unique_keywords = df_result_sentence['재군집_키워드'].dropna().unique().tolist()
print(f"추출된 고유 키워드 개수: {len(unique_keywords)}개")

template_sentences = [f"이 카페의 핵심 특징은 {kw}입니다." for kw in unique_keywords]

print("문장 임베딩 추출 중...")
sentence_embeddings = model.encode(
    template_sentences,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True  # ✅ 추가: cosine similarity 계산 시 정규화 필요
)

print("UMAP 차원 축소 중...")
umap_model = umap.UMAP(
    n_neighbors=15,  # ✅ 수정: 941개 기준으로 3은 너무 작음 → 15 권장
    n_components=5,
    metric='cosine',
    random_state=42
)
umap_embeddings = umap_model.fit_transform(sentence_embeddings)

print("HDBSCAN 클러스터링 중...")
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=3,
    metric='euclidean',
    cluster_selection_method='eom'
)
clusters = hdbscan_model.fit_predict(umap_embeddings)

df_recluster_result = pd.DataFrame({  # ✅ 수정: 기존 df_clusters_sentence 덮어쓰면 혼란스러움
    '원본_키워드': unique_keywords,
    '확장_문장(참고용)': template_sentences,
    '재군집_ID': clusters
})

print(f"\n[재군집 결과]")
print(f"  총 키워드       : {len(unique_keywords)}개")
print(f"  군집 수         : {df_recluster_result[df_recluster_result['재군집_ID'] != -1]['재군집_ID'].nunique()}개")
print(f"  노이즈(-1) 수   : {(df_recluster_result['재군집_ID'] == -1).sum()}개")
print(df_recluster_result[['원본_키워드', '재군집_ID']].sort_values(by='재군집_ID'))

/workspace/cafe_recomendation/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/cafe_recomendation/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


현재 사용 중인 디바이스: CPU
모델 로딩 중...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 401.87it/s, Materializing param=pooler.dense.weight]                               


추출된 고유 키워드 개수: 941개
문장 임베딩 추출 중...


Batches: 100%|██████████| 4/4 [00:39<00:00,  9.87s/it]
/workspace/cafe_recomendation/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP 차원 축소 중...
HDBSCAN 클러스터링 중...

[재군집 결과]
  총 키워드       : 941개
  군집 수         : 80개
  노이즈(-1) 수   : 264개
          원본_키워드  재군집_ID
490   센스가 가득한 카페      -1
290        같이 주문      -1
287  사진 찍기 좋은 카페      -1
502       화려한 간판      -1
501        해장 음료      -1
..           ...     ...
574          조아용      79
715           조식      79
618         자이푸르      79
844           조도      79
938        그릭요거트      79

[941 rows x 2 columns]


In [ ]:
df_result_sentence = df_result_sentence.merge(
    df_recluster_result[['원본_키워드', '재군집_ID']].rename(columns={'원본_키워드': '재군집_키워드'}),
    on='재군집_키워드',
    how='left'
)

(3139, 6)
      원본_키워드                  확장_문장(참고용)  군집_ID(문장기준)   대표_키워드  재군집_키워드  \
0        강아지        이 카페의 핵심 특징은 강아지입니다.           96  반려동물 카페  반려동물 카페   
1       여자친구       이 카페의 핵심 특징은 여자친구입니다.          180     여자친구     여자친구   
2       인테리어       이 카페의 핵심 특징은 인테리어입니다.          359  카페 인테리어  카페 인테리어   
3  주말마다 가고싶다  이 카페의 핵심 특징은 주말마다 가고싶다입니다.          279    주말 카페    주말 카페   
4         카페         이 카페의 핵심 특징은 카페입니다.          369      NaN       카페   

   재군집_ID  
0       4  
1      15  
2      64  
3      -1  
4      77  


In [ ]:
# df_result_sentence.to_csv('../data/ner_keyword_sent_re_clt.csv',index=False, encoding="utf-8-sig")

# 재군집 중심 키워드 추출

In [ ]:
import pandas as pd
df_result_sentence = pd.read_csv("../data/ner_keyword_sent_re_clt.csv")

In [ ]:
import json
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
from prompt import cluter_system_prompt

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def get_representative(keywords) -> str | None:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": cluter_system_prompt},
            {"role": "user", "content": str(keywords)}
        ],
        response_format={"type": "json_object"}
    )
    parsed = json.loads(response.choices[0].message.content)
    return parsed["representative"]

def get_recluster_representatives(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1차 실패 조건: 대표_키워드 None 이거나 1차 군집_ID -1
    was_first_fail = (
        df['대표_키워드'].isna() |
        (df['군집_ID(문장기준)'] == -1)
    )

    # 2차에서 군집으로 들어간 것 (재군집_ID 존재하고 -1 아닌 것)
    target_mask = (
        was_first_fail &
        df['재군집_ID'].notna() &
        (df['재군집_ID'] != -1)
    )

    # 대상 확인
    target_df = df[target_mask]
    n_clusters = target_df['재군집_ID'].nunique()
    print(f"LLM 처리 대상: {len(target_df)}개 행 / {n_clusters}개 군집")

    # 재군집_ID별 원본_키워드 묶기
    grouped = target_df.groupby('재군집_ID')['원본_키워드'].apply(list)

    # LLM 대표값 생성
    results = {}
    for cluster_id, keywords in tqdm(grouped.items(), desc="재군집 대표값 생성"):
        results[cluster_id] = get_representative(keywords)

    # 통계
    success = sum(v is not None for v in results.values())
    fail = sum(v is None for v in results.values())
    print(f"\n{'='*40}")
    print(f"  처리 군집 수  : {len(results)}개")
    print(f"  대표값 성공   : {success}개 ({success/len(results)*100:.1f}%)")
    print(f"  None (비일관) : {fail}개 ({fail/len(results)*100:.1f}%)")
    print(f"{'='*40}\n")

    # df에 재군집_대표값 컬럼 추가
    df['재군집_대표값'] = df['재군집_ID'].map(results)

    return df

df_result_sentence = get_recluster_representatives(df_result_sentence)

LLM 처리 대상: 465개 행 / 79개 군집


재군집 대표값 생성: 79it [00:56,  1.40it/s]


  처리 군집 수  : 79개
  대표값 성공   : 44개 (55.7%)
  None (비일관) : 35개 (44.3%)



In [ ]:
def make_fuzzy_keyword(row):
    # 1차 성공: 대표_키워드 있고 군집_ID -1 아닌 것
    if pd.notna(row['대표_키워드']) and row['군집_ID(문장기준)'] != -1:
        return row['대표_키워드']
    
    # 2차 성공: 재군집_대표값 있고 None 아닌 것
    if pd.notna(row['재군집_대표값']) and row['재군집_대표값'] != 'None':
        return row['재군집_대표값']
    
    # 둘 다 실패 → 원본_키워드
    return row['원본_키워드']

df_result_sentence['퍼지매칭_키워드'] = df_result_sentence.apply(make_fuzzy_keyword, axis=1)

# 원본_키워드 + 퍼지매칭_키워드만 추출
df_fuzzy = df_result_sentence[['원본_키워드', '퍼지매칭_키워드']].copy()

# 통계
total = len(df_fuzzy)
from_1st = ((df_result_sentence['대표_키워드'].notna()) & (df_result_sentence['군집_ID(문장기준)'] != -1)).sum()
from_2nd = (df_result_sentence['퍼지매칭_키워드'] == df_result_sentence['재군집_대표값']).sum()
from_orig = (df_result_sentence['퍼지매칭_키워드'] == df_result_sentence['원본_키워드']).sum()

print(f"{'='*40}")
print(f"  전체            : {total}개")
print(f"  1차 대표값      : {from_1st}개 ({from_1st/total*100:.1f}%)")
print(f"  2차 대표값      : {from_2nd}개 ({from_2nd/total*100:.1f}%)")
print(f"  원본 키워드     : {from_orig}개 ({from_orig/total*100:.1f}%)")
print(f"{'='*40}")

print(df_fuzzy.head(10))

  전체            : 3139개
  1차 대표값      : 2504개 (79.8%)
  2차 대표값      : 300개 (9.6%)
  원본 키워드     : 626개 (19.9%)
            원본_키워드         퍼지매칭_키워드
0              강아지          반려동물 카페
1             여자친구             여자친구
2             인테리어          카페 인테리어
3        주말마다 가고싶다            주말 카페
4               카페               카페
5  소량으로 판매하는게 아쉬워요  소량으로 판매하는게 아쉬워요
6              가성비           가성비 카페
7               매장               매장
8               좌석               좌석
9               조명           조명 분위기


In [ ]:
import pandas as pd
import json

with open('../data/ner_result.json', 'r', encoding='utf-8') as file:
    data = json.load(file)
df = pd.read_csv("../data/shinline_cafe_reviews_test.csv")

In [ ]:
import pandas as pd
import ast

df_store = df
ner_data = data

# ── STEP 1. df_final 생성 (MENU 포함) ─────────────
parsed_rows = []
target_categories = ['MENU', 'FACILITY', 'ATMOSPHERE', 'TARGET']  # ✅ MENU 추가

for str_idx, val in ner_data.items():
    target_idx = int(val['index'])
    
    if target_idx not in df_store.index:
        continue
        
    store_name = df_store.loc[target_idx, '사업장명']
    ner_tags = val.get('ner_tags', {})
    
    for category in target_categories:
        items = ner_tags.get(category, [])
        
        if not isinstance(items, list):
            continue
            
        for item in items:
            if isinstance(item, str):
                item = item.strip()
                if item.startswith('{') and item.endswith('}'):
                    try:
                        item = ast.literal_eval(item)
                    except (ValueError, SyntaxError):
                        continue
                else:
                    parsed_rows.append({
                        '사업장명': store_name,
                        '키워드': item,
                        '키워드타입': category,
                        'descriptor': '',
                        'sentiment': ''
                    })
                    continue
            
            if isinstance(item, dict):
                parsed_rows.append({
                    '사업장명': store_name,
                    '키워드': item.get('entity', ''),
                    '키워드타입': category,
                    'descriptor': item.get('descriptor', ''),
                    'sentiment': item.get('sentiment', '')
                })

df_final = pd.DataFrame(parsed_rows)
print(f"df_final 생성 완료: {len(df_final)}행")

# ── STEP 2. df_fuzzy 매핑 딕셔너리 생성 ───────────
# 원본_키워드 → 퍼지매칭_키워드
fuzzy_map = dict(zip(df_fuzzy['원본_키워드'], df_fuzzy['퍼지매칭_키워드']))

# ── STEP 3. 퍼지매칭 컬럼 추가 ────────────────────
# df_fuzzy에 있으면 퍼지매칭_키워드, 없으면 원본 키워드 그대로
df_final['퍼지매칭'] = df_final['키워드'].map(fuzzy_map).fillna(df_final['키워드'])

# ── 통계 ──────────────────────────────────────────
total = len(df_final)
matched = df_final['키워드'].isin(fuzzy_map).sum()
unmatched = total - matched

print(f"{'='*40}")
print(f"  전체 키워드       : {total}개")
print(f"  퍼지매칭 히트     : {matched}개 ({matched/total*100:.1f}%)")
print(f"  원본 그대로       : {unmatched}개 ({unmatched/total*100:.1f}%)")
print(f"{'='*40}")

df_final 생성 완료: 26468행
  전체 키워드       : 26468개
  퍼지매칭 히트     : 16483개 (62.3%)
  원본 그대로       : 9985개 (37.7%)


# 메뉴 키워드 처리

In [ ]:
MENU_NORMALIZE_SYSTEM_PROMPT = """
You are an expert in Korean food and beverage culture with deep knowledge of cafe and restaurant menus, including Korean slang and abbreviations used in cafe culture.
 
### [Goal]
Given a list of Korean cafe menu keywords, normalize each keyword to its canonical real-world menu name using your knowledge of food and beverages.
 
---
 
### [Output Values]
 
Each keyword must return one of the following:
1. **Canonical menu name** (string): the universally recognized name of the menu item.
2. **null**: if the keyword is NOT a menu item (see R0).
 
---
 
### [Rules]
 
**R0. Return null for non-menu keywords**
Return null if the keyword is:
- A sensory descriptor or adjective: 맛, 향, 식감, 달달한, 고소한
- A raw ingredient or component: 원두, 우유, 시럽, 크림, 얼음
- A vague evaluative word with no specific item: 음식, 메뉴, 맛집
- Anything that is not an orderable item from a cafe menu
- Examples: 맛 → null / 원두 → null / 우유 → null
 
**R1. Keep category-level words as-is**
The following broad category words are valid — return them exactly as given:
커피, 음료, 디저트, 빵, 푸드, 주류
- 커피 → 커피
- 디저트 → 디저트
 
**R2. Resolve Korean cafe slang and abbreviations**
Use your knowledge of Korean cafe culture to expand abbreviations to their full canonical name.
- 아아 → 아이스아메리카노
- 뜨아 → 아메리카노
- 아라 → 아이스라떼
- 바라 → 바닐라라떼
- 얼죽아 → 아이스아메리카노
- 아샤추 → 아이스샤인머스캣청
- 슈크림 → 슈크림빵
Apply this rule broadly — if a keyword is clearly a well-known Korean cafe slang or abbreviation, expand it.
 
**R3. Remove modifier prefixes — modifier type determines whether to remove**
 
**(3-1) ALWAYS remove these modifier types:**
- Temperature: 아이스, 핫, 따뜻한, 뜨거운, 차가운, 냉
- Intensity/degree: 라이트, 진한, 연한, 달달한, 고소한
- Style/origin: 클래식, 오리지널, 수제, 홈메이드, 프리미엄
- Season/event: 크리스마스, 시즌, 한정, 스페셜
- Store-specific branding: 시그니처, 인생, 대표
- Examples:
  - 아이스라떼 → 라떼
  - 라이트바닐라라떼 → 바닐라라떼
  - 따뜻한 라떼 → 라떼
  - 수제맥주 → 맥주
  - 크리스마스 쿠키 → 쿠키
  - 클래식 스콘 → 스콘
  - 시그니처라떼 → 라떼
  - 인생베이글 → 베이글
 
**(3-2) Keep ingredient modifiers ONLY IF the ingredient meaningfully defines a distinct menu item**
- Keep when the ingredient creates a genuinely different menu:
  - 바닐라라떼 → 바닐라라떼 (바닐라 = distinct flavor, keep)
  - 초코스콘 → 초코스콘 (초코 = distinct variant, keep)
  - 딸기라떼 → 딸기라떼 (딸기 = distinct flavor, keep)
  - 자몽주스 → 자몽주스 (자몽 = defines the juice, keep)
  - 바닐라빈라떼 → 바닐라빈라떼 (바닐라빈 = distinct ingredient, keep)
- Remove when the ingredient is incidental and does not define the item:
  - 꿀아메리카노 → 아메리카노 (꿀 = topping, does not define the item)
  - 흑당버블티 → 버블티 (흑당 = syrup variant, does not define the item)
 
**R4. Normalize spacing**
- Remove unnecessary spaces within a menu name.
  - 자몽 주스 → 자몽주스
  - 쪽파 베이글 → 쪽파베이글
 
**R5. Keep the name if it is already canonical**
If the keyword is already a well-known, universally recognized menu name, return it as-is.
- 아메리카노 → 아메리카노
- 소금빵 → 소금빵
- 마카롱 → 마카롱
- 까눌레 → 까눌레
- 밀크티 → 밀크티
 
**R6. Do not over-generalize**
Do NOT strip to a category level word (except R1).
- 소금빵 → 빵 (X)
- 아메리카노 → 커피 (X)
- 라떼 → 커피 (X)
 
**R7. Output completeness**
Every keyword in the input list must appear in the output. Do not skip or omit any keyword.
 
---
 
### [Output Format]
 
Return a single JSON object:
- Key: input keyword (string)
- Value: canonical menu name (string) or null
 
No explanation, no extra text outside the JSON block.
 
---
 
### [Examples]
 
User: ["아아", "얼죽아", "아이스라떼", "라이트바닐라라떼", "바닐라빈라떼", "꿀아메리카노", "수제맥주", "크리스마스 쿠키", "클래식 스콘", "초코스콘", "자몽 주스", "쪽파베이글", "딸기퐁당우유", "인생베이글", "시그니처라떼", "맛", "원두", "커피", "디저트", "모닝커피"]
Assistant:
{
  "아아": "아이스아메리카노",
  "얼죽아": "아이스아메리카노",
  "아이스라떼": "라떼",
  "라이트바닐라라떼": "바닐라라떼",
  "바닐라빈라떼": "바닐라빈라떼",
  "꿀아메리카노": "아메리카노",
  "수제맥주": "맥주",
  "크리스마스 쿠키": "쿠키",
  "클래식 스콘": "스콘",
  "초코스콘": "초코스콘",
  "자몽 주스": "자몽주스",
  "쪽파베이글": "쪽파베이글",
  "딸기퐁당우유": "딸기우유",
  "인생베이글": "베이글",
  "시그니처라떼": "라떼",
  "맛": null,
  "원두": null,
  "커피": "커피",
  "디저트": "디저트",
  "모닝커피": "아메리카노"
}
"""

In [ ]:
import json
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import time
#from prompt import MENU_NORMALIZE_SYSTEM_PROMPT

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ── LLM 호출 ──────────────────────────────────────────────────────────────────
def call_llm(keywords: list, retries=3) -> dict:
    user_prompt = json.dumps(keywords, ensure_ascii=False)
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="gpt-5.1",
                messages=[
                    {"role": "system", "content": MENU_NORMALIZE_SYSTEM_PROMPT},
                    {"role": "user",   "content": user_prompt}
                ],
                temperature=0,
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            print(f"  [Retry {attempt+1}/{retries}] {e}")
            time.sleep(2)
    return {kw: kw for kw in keywords}
 
 
# ── 배치 분할 ─────────────────────────────────────────────────────────────────
def make_batches(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]
 
 
# ── 메인 정규화 파이프라인 ────────────────────────────────────────────────────
def normalize_menus(df_final, batch_size=100):
    to_normalize = df_final[df_final['키워드타입'] == 'MENU']['키워드'].unique().tolist()
    print(f"정규화 대상: {len(to_normalize)}개")
 
    normalize_map = {}
    batches = list(make_batches(to_normalize, batch_size))
 
    for i, chunk in enumerate(batches):
        print(f"[Batch {i+1}/{len(batches)}] {len(chunk)}개 처리 중...")
        result = call_llm(chunk)
        normalize_map.update(result)
        time.sleep(0.5)
 
    null_count = sum(1 for v in normalize_map.values() if v is None)
    print(f"\n정규화 완료: {len(normalize_map)}개 매핑 생성 (null: {null_count}개)")
    return normalize_map
 
 
# ── df_final에 컬럼 추가 및 null 필터링 ──────────────────────────────────────
def apply_normalization(df_final, normalize_map):
    df_final['메뉴_정규화'] = df_final.apply(
        lambda r: normalize_map.get(r['키워드'], r['키워드'])
        if r['키워드타입'] == 'MENU' else r['퍼지매칭'],
        axis=1
    )
    before = len(df_final)
    df_final = df_final[
        ~((df_final['키워드타입'] == 'MENU') & (df_final['메뉴_정규화'].isnull()))
    ].reset_index(drop=True)
    print(f"null 필터링: {before - len(df_final)}행 제거 → 최종 {len(df_final)}행")
    return df_final
 
 
# ── 실행 ─────────────────────────────────────────────────────────────────────
normalize_map = normalize_menus(df_final, batch_size=100)
df_final = apply_normalization(df_final, normalize_map)
 
print("\n[샘플 확인]")
print(
    df_final[df_final['키워드타입'] == 'MENU'][['키워드', '메뉴_정규화']]
    .drop_duplicates()
    .head(30)
    .to_string(index=False)
)

정규화 대상: 2509개
[Batch 1/26] 100개 처리 중...
[Batch 2/26] 100개 처리 중...
[Batch 3/26] 100개 처리 중...
[Batch 4/26] 100개 처리 중...
[Batch 5/26] 100개 처리 중...
[Batch 6/26] 100개 처리 중...
[Batch 7/26] 100개 처리 중...
[Batch 8/26] 100개 처리 중...
[Batch 9/26] 100개 처리 중...
[Batch 10/26] 100개 처리 중...
[Batch 11/26] 100개 처리 중...
[Batch 12/26] 100개 처리 중...
[Batch 13/26] 100개 처리 중...
[Batch 14/26] 100개 처리 중...
[Batch 15/26] 100개 처리 중...
[Batch 16/26] 100개 처리 중...
[Batch 17/26] 100개 처리 중...
[Batch 18/26] 100개 처리 중...
[Batch 19/26] 100개 처리 중...
[Batch 20/26] 100개 처리 중...
[Batch 21/26] 100개 처리 중...
[Batch 22/26] 100개 처리 중...
[Batch 23/26] 100개 처리 중...
[Batch 24/26] 100개 처리 중...
[Batch 25/26] 100개 처리 중...
[Batch 26/26] 9개 처리 중...

정규화 완료: 2507개 매핑 생성 (null: 37개)
null 필터링: 67행 제거 → 최종 24026행

[샘플 확인]
       키워드    메뉴_정규화
       레몬청       레몬청
        커피        커피
       샐러드       샐러드
    쿠앤크케이크  쿠키앤크림케이크
      모크커피      모카커피
    모크크림커피    모카크림커피
   시그니처 커피        커피
     레몬머틀티     레몬머틀티
       러스크       러스크
    오레오케이크    

In [ ]:
# 현재 상태 확인
print(df_final.columns.tolist())
print(df_final.head())
print(df_final['키워드타입'].value_counts())

['사업장명', '키워드', '키워드타입', 'descriptor', 'sentiment', '퍼지매칭', '메뉴_정규화']
      사업장명   키워드   키워드타입   descriptor sentiment     퍼지매칭   메뉴_정규화
0   슬지로운생활   레몬청    MENU  담백하고 새콤달콤하다        긍정      레몬청      레몬청
1  카페캘리 본점    커피    MENU          맛있다        긍정       커피       커피
2  카페캘리 본점   강아지  TARGET           좋다        긍정  반려동물 카페  반려동물 카페
3  카페캘리 본점   샐러드    MENU          맛있다        긍정      샐러드      샐러드
4  카페캘리 본점  여자친구  TARGET      맛있다고 하다        긍정     여자친구     여자친구
키워드타입
MENU          13410
ATMOSPHERE     4062
TARGET         3452
FACILITY       3102
Name: count, dtype: int64


In [ ]:
df_final['최종_키워드'] = df_final.apply(
    lambda r: normalize_map.get(r['키워드'], r['키워드'])
              if r['키워드타입'] == 'MENU' else r['퍼지매칭'],
    axis=1
)

In [ ]:
# df_final.to_csv('../data/keyword.csv',index=False, encoding="utf-8-sig")

# 인사이트 얻기 및 최종 키워드 만들기

In [119]:
import pandas as pd
df_final = pd.read_csv('../data/keyword.csv')

In [120]:
import pandas as pd
import numpy as np
from collections import Counter

MIN_TOTAL = 10
target_types = ['MENU', 'FACILITY', 'ATMOSPHERE', 'TARGET']

# MENU 포괄어 제외
MENU_STOPWORDS = {'커피', '음료', '디저트', '빵', '푸드', '주류'}

# ── STEP 1. 키워드 생성 ───────────────────────────────────────────
def make_keyword(row):
    ktype = row['키워드타입']
    keyword = row['최종_키워드']
    descriptor = row['descriptor']
    
    if ktype in ['FACILITY', 'MENU']:
        if pd.notna(descriptor) and descriptor != '':
            return f"{keyword}_{descriptor}"
        return keyword
    else:
        return keyword

df_final['item'] = df_final.apply(make_keyword, axis=1)

# ── STEP 2. 매장별 키워드타입별 top10 집계 ───────────────────────
results_top10 = {}

for ktype in target_types:
    df_type = df_final[
        (df_final['키워드타입'] == ktype) &
        (df_final['sentiment'] == '긍정')
    ].copy()

    # MENU 포괄어 제외
    if ktype == 'MENU':
        df_type = df_type[~df_type['최종_키워드'].isin(MENU_STOPWORDS)]

    # 매장별 전체 긍정 키워드 수 (원본 기준)
    total_per_store = (
        df_type.groupby('사업장명')
        .size()
        .reset_index(name='total')
    )

    # 매장 + 최종_키워드 + item 기준 count
    counts = (
        df_type.groupby(['사업장명', '최종_키워드', 'item'])
        .size()
        .reset_index(name='count')
    )

    # 최종_키워드 기준 최빈 item만 유지
    counts = (
        counts.sort_values('count', ascending=False)
        .groupby(['사업장명', '최종_키워드'])
        .first()
        .reset_index()
    )

    counts = counts.merge(total_per_store, on='사업장명')
    counts['pct'] = (counts['count'] / counts['total'] * 100).round(1)

    counts_above = counts[counts['total'] >= MIN_TOTAL].copy()
    counts_below = counts[counts['total'] < MIN_TOTAL].copy()

    top10_above = (
        counts_above.sort_values('pct', ascending=False)
        .groupby('사업장명')
        .head(10)
        .reset_index(drop=True)
    )
    top10_above['기준'] = 'pct'

    top10_below = (
        counts_below.sort_values('count', ascending=False)
        .groupby('사업장명')
        .head(10)
        .reset_index(drop=True)
    )
    top10_below['기준'] = 'count'

    results_top10[ktype] = pd.concat([top10_above, top10_below], ignore_index=True)
    print(f"[{ktype}] top10 집계 완료: {results_top10[ktype]['사업장명'].nunique()}개 매장")

# ── STEP 3. 키워드타입별 TF-IDF ──────────────────────────────────
tfidf_results = {}

for ktype in target_types:
    df_top10 = results_top10[ktype]

    store_keywords = (
        df_top10.groupby('사업장명')
        .apply(lambda x: dict(zip(x['item'], x['count'])), include_groups=False)
        .to_dict()
    )

    total_stores = len(store_keywords)

    keyword_doc_count = Counter()
    for kws in store_keywords.values():
        for kw in kws:
            keyword_doc_count[kw] += 1

    idf = {kw: np.log(total_stores / cnt) + 1
           for kw, cnt in keyword_doc_count.items()}

    tfidf_top = []
    for store, kws in store_keywords.items():
        total_count = sum(kws.values())
        for kw, cnt in kws.items():
            tf = cnt / total_count
            score = tf * idf[kw]
            tfidf_top.append({
                '사업장명': store,
                '키워드타입': ktype,
                '키워드': kw,
                'tfidf_score': round(score, 4)
            })

    df_tfidf = (
        pd.DataFrame(tfidf_top)
        .sort_values('tfidf_score', ascending=False)
        .groupby('사업장명')
        .head(5)
        .reset_index(drop=True)
    )
    tfidf_results[ktype] = df_tfidf
    print(f"[{ktype}] TF-IDF 완료")

# ── STEP 4. 전체 합치기 ──────────────────────────────────────────
df_tfidf_all = pd.concat(tfidf_results.values(), ignore_index=True)
print(f"\n전체: {len(df_tfidf_all)}행")
print(f"매장 수: {df_tfidf_all['사업장명'].nunique()}개")

# ── 샘플 확인 ─────────────────────────────────────────────────────
store_name = '카페레이브'
print(f"\n[{store_name}] 샘플")
print(
    df_tfidf_all[df_tfidf_all['사업장명'] == store_name]
    .sort_values(['키워드타입', 'tfidf_score'], ascending=[True, False])
    .to_string(index=False)
)

[MENU] top10 집계 완료: 192개 매장
[FACILITY] top10 집계 완료: 171개 매장
[ATMOSPHERE] top10 집계 완료: 186개 매장
[TARGET] top10 집계 완료: 187개 매장
[MENU] TF-IDF 완료
[FACILITY] TF-IDF 완료
[ATMOSPHERE] TF-IDF 완료
[TARGET] TF-IDF 완료

전체: 3065행
매장 수: 201개

[카페레이브] 샘플
 사업장명      키워드타입           키워드  tfidf_score
카페레이브 ATMOSPHERE        카페 분위기       0.6578
카페레이브 ATMOSPHERE        아늑한 카페       0.3477
카페레이브 ATMOSPHERE       코지한 분위기       0.2285
카페레이브 ATMOSPHERE        조용한 카페       0.2183
카페레이브 ATMOSPHERE        다락방 느낌       0.1706
카페레이브   FACILITY   카페 인테리어_예쁘다       0.8083
카페레이브   FACILITY        좌석_편하다       0.5720
카페레이브   FACILITY       식기류_예쁘다       0.4953
카페레이브   FACILITY        카페_이쁘다       0.3954
카페레이브   FACILITY       매장_깔끔하다       0.3345
카페레이브       MENU       케이크_맛있다       1.0782
카페레이브       MENU      쑥케이크_맛있다       0.7065
카페레이브       MENU       쑥라떼_맛있다       0.5385
카페레이브       MENU 얼그레이초코케이크_맛있다       0.5046
카페레이브       MENU    고구마케이크_맛있다       0.3328
카페레이브     TARGET        데이트 장소       0.6821
카페레이브     TARG

In [121]:
df_tfidf_all

,사업장명,키워드타입,키워드,tfidf_score
0,더벤티 서울대점,MENU,샌드위치_가격대비 괜찮다,6.2575
1,카페마실,MENU,마카롱_가성비 있다,6.2575
2,또리커피 관악점,MENU,아이스아메리카노_양이 많다,6.2575
3,컴포즈커피 신림신화교점,MENU,망고케이크_맛있다,5.1589
4,엣홈,MENU,휘낭시에_맛있다,3.5494
...,...,...,...,...
3060,카페레이브,TARGET,반려동물 카페,0.2373
3061,카페레이브,TARGET,단골카페,0.2361
3062,별별카롱,TARGET,행사때 구입하기 좋은 것,0.2010
3063,카페디딤,TARGET,혼자 작업,0.1926


In [122]:
## 부정 키워드 추출
neg_final = {}

for ktype in ['FACILITY', 'ATMOSPHERE', 'MENU']:
    df_neg = df_final[
        (df_final['키워드타입'] == ktype) &
        (df_final['sentiment'] == '부정')
    ].copy()

    # 불용어 제거
    stopwords = NEG_STOPWORDS.get(ktype, set())
    df_neg = df_neg[~df_neg['최종_키워드'].isin(stopwords)]

    # 매장별 키워드 count
    counts = (
        df_neg.groupby(['사업장명', '최종_키워드'])
        .size()
        .reset_index(name='count')
    )

    # global_count 계산
    global_neg = (
        counts.groupby('최종_키워드')['count']
        .sum()
        .reset_index(name='global_count')
    )

    counts = counts.merge(global_neg, on='최종_키워드')

    # count > 1, global_count > 1 필터링
    counts = counts[
        (counts['count'] > 1) &
        (counts['global_count'] > 1)
    ]

    # 1차: count, 2차: global_count top3
    top3 = (
        counts.sort_values(['count', 'global_count'], ascending=[False, False])
        .groupby('사업장명')
        .head(3)
        .reset_index(drop=True)
    )
    top3['키워드타입'] = ktype
    neg_final[ktype] = top3

df_neg_final = pd.concat(neg_final.values(), ignore_index=True)

print(f"전체: {len(df_neg_final)}행")
print(f"매장 수: {df_neg_final['사업장명'].nunique()}개")

for ktype in ['FACILITY', 'ATMOSPHERE', 'MENU']:
    df_k = df_neg_final[df_neg_final['키워드타입'] == ktype]
    print(f"\n[{ktype} 부정]")
    print(f"  매장 수: {df_k['사업장명'].nunique()}개")
    print(df_k.to_string(index=False))

전체: 87행
매장 수: 45개

[FACILITY 부정]
  매장 수: 14개
          사업장명  최종_키워드  count  global_count    키워드타입
   투썸플레이스 신림로점     화장실      9            31 FACILITY
     이디야 신림미성점    와이파이      7            13 FACILITY
파스쿠찌 서울대관정도서관점    와이파이      4            13 FACILITY
    빽다방 신림역2호점 내부 인테리어      4             7 FACILITY
          어반커피     화장실      3            31 FACILITY
     이디야 신림미성점     화장실      3            31 FACILITY
     이디야 신림미성점   2층 카페      3             9 FACILITY
탐앤탐스 삼모타워 신림역점     흡연실      3             9 FACILITY
  할리스커피 난곡사거리점     흡연실      3             9 FACILITY
  카페 아래로 ARERO      주차      3             8 FACILITY
  네이밍카페커피GO!#1     화장실      2            31 FACILITY
         쉬즈베이글     화장실      2            31 FACILITY
파스쿠찌 서울대관정도서관점     화장실      2            31 FACILITY
  파스쿠찌 센트로서울대점     화장실      2            31 FACILITY
       할리스 신림점     화장실      2            31 FACILITY
탐앤탐스 삼모타워 신림역점     테이블      2            20 FACILITY
투썸플레이스 신림녹두거리점      좌석      2            20 FACILITY
 

In [123]:
# 긍정에 sentiment 컬럼 추가
df_tfidf_all['sentiment'] = '긍정'
df_tfidf_all = df_tfidf_all.rename(columns={'키워드': '최종_키워드'})

# 부정에 sentiment 컬럼 추가
df_neg_final['sentiment'] = '부정'

# 컬럼 통일
df_pos = df_tfidf_all[['사업장명', '키워드타입', '최종_키워드', 'tfidf_score', 'sentiment']]
df_neg = df_neg_final[['사업장명', '키워드타입', '최종_키워드', 'count', 'global_count', 'sentiment']]

# 합치기
df_store_items = pd.concat([df_pos, df_neg], ignore_index=True)

print(f"전체: {len(df_store_items)}행")
print(f"매장 수: {df_store_items['사업장명'].nunique()}개")

# 샘플 확인
store_name = '카페레이브'
print(f"\n[{store_name}] 최종 item")
print(
    df_store_items[df_store_items['사업장명'] == store_name]
    .sort_values(['sentiment', '키워드타입'], ascending=[False, True])
    .to_string(index=False)
)

전체: 3152행
매장 수: 201개

[카페레이브] 최종 item
 사업장명      키워드타입        최종_키워드  tfidf_score sentiment  count  global_count
카페레이브 ATMOSPHERE        카페 분위기       0.6578        긍정    NaN           NaN
카페레이브 ATMOSPHERE        아늑한 카페       0.3477        긍정    NaN           NaN
카페레이브 ATMOSPHERE       코지한 분위기       0.2285        긍정    NaN           NaN
카페레이브 ATMOSPHERE        조용한 카페       0.2183        긍정    NaN           NaN
카페레이브 ATMOSPHERE        다락방 느낌       0.1706        긍정    NaN           NaN
카페레이브   FACILITY   카페 인테리어_예쁘다       0.8083        긍정    NaN           NaN
카페레이브   FACILITY        좌석_편하다       0.5720        긍정    NaN           NaN
카페레이브   FACILITY       식기류_예쁘다       0.4953        긍정    NaN           NaN
카페레이브   FACILITY        카페_이쁘다       0.3954        긍정    NaN           NaN
카페레이브   FACILITY       매장_깔끔하다       0.3345        긍정    NaN           NaN
카페레이브       MENU       케이크_맛있다       1.0782        긍정    NaN           NaN
카페레이브       MENU      쑥케이크_맛있다       0.7065        긍정    NaN  

In [126]:
df_store_items.to_csv('../data/market_item.csv',index=False, encoding="utf-8-sig")

In [125]:
df_store_items

,사업장명,키워드타입,최종_키워드,tfidf_score,sentiment,count,global_count
0,더벤티 서울대점,MENU,샌드위치_가격대비 괜찮다,6.2575,긍정,NaN,NaN
1,카페마실,MENU,마카롱_가성비 있다,6.2575,긍정,NaN,NaN
2,또리커피 관악점,MENU,아이스아메리카노_양이 많다,6.2575,긍정,NaN,NaN
3,컴포즈커피 신림신화교점,MENU,망고케이크_맛있다,5.1589,긍정,NaN,NaN
4,엣홈,MENU,휘낭시에_맛있다,3.5494,긍정,NaN,NaN
...,...,...,...,...,...,...,...
3147,우체국 옆 빵집,MENU,크루아상,NaN,부정,2.0,2.0
3148,이디야 신림미성점,MENU,라이스치즈볼,NaN,부정,2.0,2.0
3149,이디야 신림미성점,MENU,카푸치노,NaN,부정,2.0,2.0
3150,투썸플레이스 신림녹두거리점,MENU,마스카포네케이크,NaN,부정,2.0,2.0


In [1]:
import pandas as pd
df_store_item = pd.read_csv('../data/market_item.csv')

In [4]:
df_store_item['최종_키워드'].unique()

array(['샌드위치_가격대비 괜찮다', '마카롱_가성비 있다', '아이스아메리카노_양이 많다', ..., '카푸치노',
       '마스카포네케이크', '소금프레첼베이글'], shape=(1425,), dtype=object)

In [10]:
len(df_store_item[df_store_item['키워드타입'] != 'MENU']['최종_키워드'].unique())

798

array(['케이크_맛있다', '팥빙수_맛있다', '티라미수_맛있다', '헤이즐넛라떼_맛있다', '로얄밀크티쉐이크_평타 이상이다',
       '매장_넓다', '자리_많다', '좌석_많다', '주차_가능하고 편리하다', '콘센트_잘 구비되어있다',
       '조용한 카페', '충전', '스누피 콜라보 커피', '눈치 주는 분위기', '한적한 분위기', '타임', '대화',
       '일기', '계획', '남자친구', '좌석', '케이크', '마스카포네케이크', '소금프레첼베이글'],
      dtype=object)